<a href="https://colab.research.google.com/github/SAMMY-4870/Data-Science-Advanced-Programming/blob/main/unit4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files

uploaded = files.upload()



In [ ]:

import pandas as pd

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:", df.isnull().sum().sum())


In [ ]:

# STEP 5: DATA PREPROCESSING

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Keep original dataset safe
data = df.copy()

# Remove unnecessary columns
columns_to_drop = [
    'EmployeeNumber',
    'EmployeeCount',
    'Over18',
    'StandardHours'
]

data = data.drop(
    columns=[c for c in columns_to_drop if c in data.columns],
    errors='ignore'
)

# Separate employee ID and attrition label for analysis
employee_ids = data['EmployeeNumber'].copy() if 'EmployeeNumber' in data.columns else pd.Series(data.index)
attrition = data['Attrition'].copy() if 'Attrition' in data.columns else None

# Remove Attrition from clustering features to avoid label leakage
if 'Attrition' in data.columns:
    data = data.drop(columns=['Attrition'])

# Remove constant columns
data = data.loc[:, data.nunique() > 1]

# Convert categorical features to numerical
X = pd.get_dummies(data, drop_first=True, dtype=int)

# Standardize all features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original dataset shape:", df.shape)
print("Features after encoding:", X.shape)
print("Scaled data shape:", X_scaled.shape)
print("Missing values:", X.isnull().sum().sum())
print("Preprocessing completed successfully!")


In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

inertia_values = []
silhouette_values = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)

    inertia_values.append(model.inertia_)
    silhouette_values.append(
        silhouette_score(
            X_scaled,
            labels,
            sample_size=min(2000, len(X_scaled)),
            random_state=42
        )
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_values), inertia_values, marker='o')
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_values), silhouette_values, marker='o')
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

print("Silhouette scores:")
for k, score in zip(k_values, silhouette_values):
    print(f"K={k}: {score:.4f}")


In [ ]:

# STEP 7: FINAL K-MEANS CLUSTERING

from sklearn.cluster import KMeans
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Choose cluster count based on Step 6 results
best_k = 4

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

# Store cluster labels with original employee data
clustered_df = df.copy()
clustered_df["KMeans_Cluster"] = cluster_labels

print("K-Means clustering completed!")
print("\nEmployees in each cluster:")
print(clustered_df["KMeans_Cluster"].value_counts().sort_index())

# Visualize cluster sizes
plt.figure(figsize=(8, 5))
sns.countplot(
    data=clustered_df,
    x="KMeans_Cluster",
    hue="KMeans_Cluster",
    palette="Set2",
    legend=False
)
plt.title("Employee Distribution Across K-Means Clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of Employees")
plt.show()


In [ ]:

# STEP 8: CLUSTER-WISE WORKFORCE ANALYSIS

clustered_df["Attrition_Numeric"] = (
    clustered_df["Attrition"].map({"Yes": 1, "No": 0})
)

numeric_cols = clustered_df.select_dtypes(
    include="number"
).columns.tolist()

# Remove identifiers and cluster labels from averages
exclude_cols = [
    "EmployeeNumber",
    "KMeans_Cluster",
    "Attrition_Numeric"
]

analysis_cols = [
    col for col in numeric_cols
    if col not in exclude_cols
]

cluster_summary = clustered_df.groupby(
    "KMeans_Cluster"
)[analysis_cols].mean().round(2)

print("Cluster-wise Employee Profile:")
display(cluster_summary)

# Attrition rate in each cluster
attrition_summary = (
    clustered_df.groupby("KMeans_Cluster")
    .agg(
        Employees=("Attrition", "size"),
        Attrition_Count=("Attrition_Numeric", "sum"),
        Attrition_Rate=("Attrition_Numeric", "mean")
    )
)

attrition_summary["Attrition_Rate"] = (
    attrition_summary["Attrition_Rate"] * 100
).round(2)

print("Cluster-wise Attrition Analysis:")
display(attrition_summary)


In [ ]:

# STEP 9: HIERARCHICAL CLUSTERING

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Use the same number of clusters as K-Means
best_k = 4

hierarchical = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)

hierarchical_labels = hierarchical.fit_predict(X_scaled)

# Save cluster assignments
clustered_df["Hierarchical_Cluster"] = hierarchical_labels

print("Hierarchical Clustering Completed!")

print("\nEmployees in each cluster:")
print(
    clustered_df["Hierarchical_Cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nSilhouette Score:",
    round(silhouette_score(X_scaled, hierarchical_labels), 4)
)

# Visualize cluster sizes
plt.figure(figsize=(8, 5))
sns.countplot(
    data=clustered_df,
    x="Hierarchical_Cluster",
    hue="Hierarchical_Cluster",
    palette="Set2",
    legend=False
)
plt.title("Employee Distribution - Hierarchical Clustering")
plt.xlabel("Cluster")
plt.ylabel("Number of Employees")
plt.show()


In [ ]:

# STEP 10: MODEL COMPARISON

from sklearn.metrics import silhouette_score

kmeans_score = silhouette_score(
    X_scaled,
    clustered_df["KMeans_Cluster"]
)

hierarchical_score = silhouette_score(
    X_scaled,
    clustered_df["Hierarchical_Cluster"]
)

comparison = pd.DataFrame({
    "Algorithm": ["K-Means", "Hierarchical Clustering"],
    "Silhouette Score": [
        round(kmeans_score, 4),
        round(hierarchical_score, 4)
    ]
})

display(comparison)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=comparison,
    x="Algorithm",
    y="Silhouette Score",
    hue="Algorithm",
    legend=False,
    palette="viridis"
)
plt.title("Clustering Algorithm Comparison")
plt.ylabel("Silhouette Score")
plt.show()


In [ ]:

# STEP 11: PCA DIMENSIONALITY REDUCTION

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "KMeans_Cluster": clustered_df["KMeans_Cluster"]
})

print(
    "Explained Variance Ratio:",
    pca.explained_variance_ratio_.round(4)
)

print(
    "Total Variance Explained:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="KMeans_Cluster",
    palette="Set2",
    alpha=0.7
)

plt.title("Employee Segmentation using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()


In [ ]:

# STEP 13: RETENTION RISK ANALYSIS

risk_df = df.copy()
risk_df["KMeans_Cluster"] = clustered_df["KMeans_Cluster"].values

# Convert historical attrition to numeric
risk_df["Attrition_Flag"] = risk_df["Attrition"].map({
    "Yes": 1,
    "No": 0
})

# Cluster-level attrition
cluster_risk = (
    risk_df.groupby("KMeans_Cluster")
    .agg(
        Employee_Count=("Attrition_Flag", "size"),
        Attrition_Count=("Attrition_Flag", "sum"),
        Attrition_Rate=("Attrition_Flag", "mean")
    )
)

cluster_risk["Attrition_Rate"] = (
    cluster_risk["Attrition_Rate"] * 100
).round(2)

print("Historical Attrition by Cluster:")
display(cluster_risk)

# Department-level attrition
dept_risk = (
    risk_df.groupby("Department")
    .agg(
        Employee_Count=("Attrition_Flag", "size"),
        Attrition_Count=("Attrition_Flag", "sum"),
        Attrition_Rate=("Attrition_Flag", "mean")
    )
)

dept_risk["Attrition_Rate"] = (
    dept_risk["Attrition_Rate"] * 100
).round(2)

print("Historical Attrition by Department:")
display(dept_risk.sort_values("Attrition_Rate", ascending=False))

dept_risk["Attrition_Rate"].sort_values().plot(
    kind="barh",
    figsize=(9, 5),
    color="coral"
)
plt.title("Historical Attrition Rate by Department")
plt.xlabel("Attrition Rate (%)")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


In [ ]:

# STEP 14: OVERTIME AND JOB SATISFACTION

# Historical attrition rate by overtime
overtime_analysis = (
    df.assign(
        Attrition_Flag=df["Attrition"].map({"Yes": 1, "No": 0})
    )
    .groupby("OverTime")["Attrition_Flag"]
    .agg(["count", "mean"])
)

overtime_analysis.columns = [
    "Employee_Count",
    "Attrition_Rate"
]
overtime_analysis["Attrition_Rate"] *= 100

print("Overtime Analysis:")
display(overtime_analysis.round(2))

# Historical attrition by job satisfaction
satisfaction_analysis = (
    df.assign(
        Attrition_Flag=df["Attrition"].map({"Yes": 1, "No": 0})
    )
    .groupby("JobSatisfaction")["Attrition_Flag"]
    .agg(["count", "mean"])
)

satisfaction_analysis.columns = [
    "Employee_Count",
    "Attrition_Rate"
]
satisfaction_analysis["Attrition_Rate"] *= 100

print("Job Satisfaction Analysis:")
display(satisfaction_analysis.round(2))

# Visualize overtime relationship
plt.figure(figsize=(7, 5))
sns.barplot(
    data=df,
    x="OverTime",
    y=df["Attrition"].eq("Yes").astype(int),
    errorbar=None
)
plt.title("Overtime vs Historical Attrition")
plt.xlabel("Overtime")
plt.ylabel("Historical Attrition Rate")
plt.tight_layout()
plt.show()


In [ ]:

# STEP 15: SAVE ANALYSIS OUTPUTS

import os

os.makedirs("reports", exist_ok=True)

clustered_df.to_csv(
    "reports/employee_cluster_assignments.csv",
    index=False
)

cluster_risk.to_csv(
    "reports/cluster_retention_analysis.csv"
)

dept_risk.to_csv(
    "reports/department_retention_analysis.csv"
)

pca_df.to_csv(
    "reports/employee_pca_coordinates.csv",
    index=False
)

print("Analysis files saved successfully!")

print("Available files:")
print(os.listdir("reports"))


In [ ]:

# STEP 16: INTERACTIVE WORKFORCE ANALYTICS DASHBOARD

import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

# Prepare dashboard data
dashboard_df = df.copy()
dashboard_df["KMeans_Cluster"] = (
    clustered_df["KMeans_Cluster"].values
)

dashboard_df["Attrition_Flag"] = (
    dashboard_df["Attrition"].map({"Yes": 1, "No": 0})
)

# KPI calculations
total_employees = len(dashboard_df)
total_departments = dashboard_df["Department"].nunique()
total_clusters = dashboard_df["KMeans_Cluster"].nunique()
overall_attrition = dashboard_df["Attrition_Flag"].mean() * 100

print("EMPLOYEE WORKFORCE SEGMENTATION DASHBOARD")
print("=" * 48)
print("Total Employees:", total_employees)
print("Departments:", total_departments)
print("Employee Clusters:", total_clusters)
print(f"Historical Attrition Rate: {overall_attrition:.2f}%")

# Chart 1: Employee distribution by cluster
cluster_counts = (
    dashboard_df["KMeans_Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="Employees")
)

fig1 = px.bar(
    cluster_counts,
    x="Cluster",
    y="Employees",
    color="Cluster",
    title="Employee Distribution by Cluster",
    text="Employees"
)
fig1.show()

# Chart 2: Department-wise workforce
dept_counts = (
    dashboard_df.groupby("Department")
    .size()
    .reset_index(name="Employees")
)

fig2 = px.bar(
    dept_counts,
    x="Department",
    y="Employees",
    color="Department",
    title="Department-wise Workforce"
)
fig2.show()

# Chart 3: PCA cluster visualization
pca_dashboard = pca_df.copy()

fig3 = px.scatter(
    pca_dashboard,
    x="PC1",
    y="PC2",
    color="KMeans_Cluster",
    title="PCA Visualization of Employee Segments",
    opacity=0.7
)
fig3.show()

# Chart 4: Historical attrition by department
dept_attrition = (
    dashboard_df.groupby("Department")
    .agg(
        Employees=("Attrition_Flag", "size"),
        Attrition_Rate=("Attrition_Flag", "mean")
    )
    .reset_index()
)

dept_attrition["Attrition_Rate"] *= 100

fig4 = px.bar(
    dept_attrition,
    x="Department",
    y="Attrition_Rate",
    color="Department",
    hover_data=["Employees"],
    title="Historical Attrition Rate by Department",
    labels={"Attrition_Rate": "Attrition Rate (%)"}
)
fig4.show()

print("Dashboard charts generated successfully!")


In [ ]:

# STEP 17: EXPORT DASHBOARD AS HTML

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Employees by Cluster",
        "Department-wise Workforce",
        "PCA Employee Segments",
        "Historical Attrition by Department"
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "bar"}]
    ]
)

# Cluster chart
for cluster in sorted(dashboard_df["KMeans_Cluster"].unique()):
    count = (
        dashboard_df["KMeans_Cluster"] == cluster
    ).sum()

    fig.add_trace(
        go.Bar(
            x=[f"Cluster {cluster}"],
            y=[count],
            name=f"Cluster {cluster}"
        ),
        row=1, col=1
    )

# Department chart
fig.add_trace(
    go.Bar(
        x=dept_counts["Department"],
        y=dept_counts["Employees"],
        name="Employees by Department"
    ),
    row=1, col=2
)

# PCA scatter plot
for cluster in sorted(pca_dashboard["KMeans_Cluster"].unique()):
    part = pca_dashboard[
        pca_dashboard["KMeans_Cluster"] == cluster
    ]

    fig.add_trace(
        go.Scatter(
            x=part["PC1"],
            y=part["PC2"],
            mode="markers",
            name=f"PCA Cluster {cluster}"
        ),
        row=2, col=1
    )

# Attrition chart
fig.add_trace(
    go.Bar(
        x=dept_attrition["Department"],
        y=dept_attrition["Attrition_Rate"],
        name="Attrition Rate (%)"
    ),
    row=2, col=2
)

fig.update_layout(
    title="Employee Workforce Segmentation Dashboard",
    height=850,
    width=1200,
    template="plotly_white",
    barmode="group"
)

fig.write_html(
    "reports/workforce_analytics_dashboard.html",
    include_plotlyjs=True
)

fig.show()

print("Dashboard exported successfully!")


In [ ]:

# STEP 19: GENERATE PROJECT REPORTS

import os
import pandas as pd
from sklearn.metrics import silhouette_score

os.makedirs("reports", exist_ok=True)

# Calculate clustering evaluation metrics
kmeans_score = silhouette_score(
    X_scaled, clustered_df["KMeans_Cluster"]
)

hierarchical_score = silhouette_score(
    X_scaled, clustered_df["Hierarchical_Cluster"]
)

# PCA statistics
variance = pca.explained_variance_ratio_
total_variance = variance.sum() * 100

# Overall workforce statistics
total_employees = len(df)
total_departments = df["Department"].nunique()
overall_attrition = df["Attrition"].eq("Yes").mean() * 100

# Build employee segmentation report
report = f"""
EMPLOYEE WORKFORCE SEGMENTATION REPORT
======================================

1. PROJECT OVERVIEW
Project: Employee Workforce Segmentation
Dataset: IBM HR Analytics Employee Attrition & Performance

2. WORKFORCE SUMMARY
Total Employees: {total_employees}
Departments: {total_departments}
K-Means Clusters: {clustered_df['KMeans_Cluster'].nunique()}
Hierarchical Clusters: {clustered_df['Hierarchical_Cluster'].nunique()}
Historical Attrition Rate: {overall_attrition:.2f}%

3. CLUSTERING EVALUATION
K-Means Silhouette Score: {kmeans_score:.4f}
Hierarchical Silhouette Score: {hierarchical_score:.4f}

Higher silhouette scores generally indicate better-separated
clusters, but cluster sizes and interpretability must also
be considered.

4. PCA SUMMARY
Principal Component 1 Variance: {variance[0] * 100:.2f}%
Principal Component 2 Variance: {variance[1] * 100:.2f}%
Combined Variance Explained: {total_variance:.2f}%

PCA reduces feature dimensions for visualization.
The two-component representation may not preserve all
information from the original dataset.

5. EMPLOYEE DISTRIBUTION
{clustered_df['KMeans_Cluster'].value_counts().sort_index().to_string()}

6. DEPARTMENT-LEVEL HISTORICAL ATTRITION
{dept_risk.to_string()}

7. CLUSTER-LEVEL HISTORICAL ATTRITION
{cluster_risk.to_string()}

8. LIMITATIONS
- This is a sample HR dataset, not live company data.
- Clusters represent statistical similarities, not verified
  employee personality or performance categories.
- Historical attrition is not a guaranteed future outcome.
- Cluster quality should be evaluated alongside business context.

END OF REPORT
"""

with open("reports/employee_segmentation_report.txt", "w",
          encoding="utf-8") as f:
    f.write(report)

# Save PCA summary
pca_summary = pd.DataFrame({
    "Component": ["PC1", "PC2"],
    "Explained_Variance_Ratio": variance,
    "Explained_Variance_Percent": variance * 100
})

pca_summary.loc[len(pca_summary)] = [
    "Combined PC1 + PC2",
    total_variance / 100,
    total_variance
]

pca_summary.to_csv(
    "reports/pca_summary.csv", index=False
)

# Save model comparison
pd.DataFrame({
    "Algorithm": ["K-Means", "Hierarchical Clustering"],
    "Silhouette_Score": [kmeans_score, hierarchical_score]
}).to_csv("reports/clustering_comparison.csv", index=False)

print("Reports generated successfully!")
print(report)


In [ ]:

# STEP 20: EXPORT PROJECT OUTPUTS

import os
import zipfile
from google.colab import files

zip_name = "employee_workforce_segmentation.zip"

# Include all reports and dashboard files
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk("reports"):
        for filename in filenames:
            path = os.path.join(root, filename)
            zf.write(path, arcname=path)

print("ZIP created:", zip_name)
print("Files included:")

with zipfile.ZipFile(zip_name, "r") as zf:
    for name in zf.namelist():
        print(name)

files.download(zip_name)
